# First scan in scanlang - Jupyter edition

Run end-to-end with:

```sh
uv run jupyter nbconvert --execute --to notebook --inplace \
  docs/notebooks/01_first_scan.ipynb
```

This notebook tests a scan on raw OHLCV bars first. It then scores the
bars and applies a second screen to the scored output.

In [1]:
import sys
from pathlib import Path

notebook_dir = Path("docs/notebooks")
if not (notebook_dir / "_fixture.py").exists():
    notebook_dir = Path(".")
sys.path.insert(0, str(notebook_dir))

import polars as pl

from _fixture import RAW_SCAN_DEF, SCORE_SCAN_DEF, bars_eager, bars_lazy
from scanlang import apply, parse, score_bars, validate

## 1. Prepare the fixture

The shared fixture contains 60 sessions for each of two symbols and
the required OHLCV columns, sorted by `symbol` and `session`.

In [2]:
df_eager = bars_eager()
df_lazy = bars_lazy()
assert df_eager.shape == (120, 7)
assert df_lazy.collect().shape == (120, 7)
df_eager.head(2)

symbol,session,open,high,low,close,volume
str,date,f64,f64,f64,f64,f64
"""AAA""",2026-01-01,9.5,11.0,9.0,10.0,1000.0
"""AAA""",2026-01-02,10.5,12.0,10.0,11.0,1000.0


## 2. Test a scan on raw bars first

This screen uses only `close`, so it can run before `score_bars`. The
text DSL parses to the same definition as the shared dict.

In [3]:
raw_scan = parse("ema(5) > ema(20)")
assert raw_scan == RAW_SCAN_DEF
raw_errors = validate(raw_scan)
assert raw_errors == []
raw_picks = apply(df_eager, raw_scan)
assert raw_picks.height == 59
assert raw_picks["symbol"].unique().to_list() == ["AAA"]
raw_picks.select("symbol", "session", "close").head()

symbol,session,close
str,date,f64
"""AAA""",2026-01-02,11.0
"""AAA""",2026-01-03,12.0
"""AAA""",2026-01-04,13.0
"""AAA""",2026-01-05,14.0
"""AAA""",2026-01-06,15.0


## 3. Score the latest bar for each symbol

Now add the derived score fields used by the second screen.
`score_bars` is lazy in and lazy out; this notebook collects at the
eager boundary.

In [4]:
scored = score_bars(df_eager).collect()
scored.select("symbol", "session", "close", "score", "phase")

symbol,session,close,score,phase
str,date,f64,i16,str
"""AAA""",2026-03-01,69.0,60,"""BASE"""
"""BBB""",2026-03-01,1.0,20,"""NONE"""


## 4. Validate and apply a scored screen

A scan definition is a plain dict. `validate` returns `[]` when it is
valid; `apply` then filters, orders, and limits the frame.

In [5]:
errors = validate(SCORE_SCAN_DEF)
assert errors == []
picks = apply(scored, SCORE_SCAN_DEF)
assert picks.height == 1
assert picks["symbol"][0] == "AAA"
picks.select("symbol", "score", "phase")

symbol,score,phase
str,i16,str
"""AAA""",60,"""BASE"""


## 5. Keep a lazy pipeline when useful

The same scored screen can stay lazy until the final display.

In [6]:
lazy_picks = apply(score_bars(df_lazy), SCORE_SCAN_DEF)
assert isinstance(lazy_picks, pl.LazyFrame)
lazy_result = lazy_picks.select("symbol", "score", "phase").collect()
assert lazy_result.height == 1
lazy_result

symbol,score,phase
str,i16,str
"""AAA""",60,"""BASE"""


## 6. Lock the behavior

Executing this notebook must fail if parsing, validation, raw scans,
scoring, or the lazy path regresses.

In [7]:
assert raw_errors == []
assert errors == []
assert raw_picks.height == 59
assert picks.height == 1
assert picks["score"][0] == scored["score"].max()
assert lazy_result.rows() == picks.select("symbol", "score", "phase").rows()
print("01_first_scan OK")

01_first_scan OK


## Where to next

- [`02_first_scan_marimo.py`](./02_first_scan_marimo.py) - the same
  raw-first workflow in a reactive notebook.
- [Use it](../use.md) - the complete library workflow.
- [Language](../language.md) - DSL and dict syntax.
- [Examples](../more.md#examples-and-notebooks) - runnable scripts.